# 単語の頻度から語彙の難しさを推定する (wordfreq)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/25_gec_education/word_frequency_level.ipynb)

NLP クックブック 25-17 ／ 文法誤り訂正と言語教育

文中のどの単語が「珍しい (難しい)」のかを、大規模コーパスでの出現頻度から見積もりたい。

In [1]:
!pip install -q "wordfreq[cjk]" fugashi unidic-lite

In [2]:
from wordfreq import top_n_list, zipf_frequency

# Zipf 尺度: 10 億語あたりの出現回数の常用対数。7 = 極めて高頻度、3 以下 = 低頻度
for w in ["the", "language", "linguistics", "morphology", "agglutinative"]:
    print(f"{w:15}{zipf_frequency(w, 'en'):.2f}")
for w in ["猫", "言語", "翻訳", "形態素", "膠着語"]:
    print(f"{w}\t{zipf_frequency(w, 'ja'):.2f}")
print(top_n_list("ja", 15))

the            7.73
language       5.10
linguistics    3.43
morphology     3.41
agglutinative  1.58
猫	5.05
言語	4.65
翻訳	4.61
形態素	1.96
膠着語	2.76
['の', 'に', 'て', 'は', 'が', 'た', 'を', 'で', 'し', 'と', 'な', 'ない', 'も', 'だ', 'か']


In [3]:
from fugashi import Tagger

tagger = Tagger()

def difficult_words(text, lang="ja", threshold=4.0):
    """内容語のうち、Zipf 頻度がしきい値未満のものを返す"""
    out = []
    for w in tagger(text):
        if w.feature.pos1 in ("名詞", "動詞", "形容詞", "形状詞", "副詞"):
            base = w.feature.orthBase or w.surface
            z = zipf_frequency(base, lang)
            if z < threshold:
                out.append((base, round(z, 2)))
    return out

for text in ["猫が庭で寝ている。", "当該事案の瑕疵について遺漏なく精査されたい。", "形態素解析器が膠着語の接辞を分離する。"]:
    print(text, difficult_words(text))

猫が庭で寝ている。 []
当該事案の瑕疵について遺漏なく精査されたい。 [('事案', 3.77), ('瑕疵', 3.08), ('遺漏', 1.77), ('精査', 3.44)]
形態素解析器が膠着語の接辞を分離する。 [('膠着', 2.76), ('接辞', 1.58)]


- wordfreq は Wikipedia、字幕、Web、SNS など複数の資源を統合した頻度表で、40 以上の言語に対応している (データは 2021 年までで更新終了)。
- 頻度は難しさの近似にすぎない。高頻度でも多義で難しい語 (「かける」「とる」) や、学習者の母語によって易しい語 (漢字圏の学習者にとっての漢語) がある。
- 日本語の教育語彙としては、日本語教育語彙表や旧日本語能力試験の出題基準などが使われる。